# ConvTransformer para detección y localización de puntos de cambio — \(L=200\)

Este cuaderno mejorado adapta el notebook `11_convtransformer_binary_detection_localization.ipynb` al caso de trayectorias de longitud \(L=200\). El objetivo es entrenar y evaluar una arquitectura fija, **ConvTransformer**, utilizando los ficheros `train/val/test` generados para \(L=200\).

Además, el cuaderno incluye el análisis solicitado por el profesor: matrices por transición ordenada, donde las **filas** representan el modelo generador de la primera parte de la trayectoria y las **columnas** el modelo generador de la segunda parte.


## Enfoque del cuaderno

Una vez identificado ConvTransformer como la arquitectura con mejor comportamiento global en el análisis previo para \(L=100\), este cuaderno fija dicha arquitectura y repite el análisis para \(L=200\).

La tarea mantiene dos salidas:

- `has_cp`: detección binaria de la presencia de punto de cambio.
- `cp_dist`: distribución de probabilidad sobre las posiciones temporales posibles del punto de cambio.

El análisis final se centra especialmente en las transiciones ordenadas \(M_1 ightarrow M_2\), con \(M_1, M_2 \in \{\mathrm{ATTM}, \mathrm{CTRW}, \mathrm{FBM}, \mathrm{LW}, \mathrm{SBM}\}\) y \(M_1 
eq M_2\).


## Objetivo específico del análisis por transiciones

Para responder a la indicación del profesor, este cuaderno genera figuras matriciales para \(L=200\), manteniendo fija la arquitectura ConvTransformer. En cada figura:

- las **filas** indican el modelo de difusión de la primera parte de la trayectoria;
- las **columnas** indican el modelo de difusión de la segunda parte;
- cada celda representa una transición ordenada \(M_1 ightarrow M_2\).

Estas matrices permiten ver, más allá de las métricas globales, entre qué pares de modelos se concentran los problemas de detección y localización.


## 1. Configuración experimental

Se definen los parámetros generales de ejecución, las rutas de entrada y salida, las semillas aleatorias y las opciones que controlan el tamaño del experimento. El modo rápido permite comprobar el flujo completo con un subconjunto equilibrado, mientras que el modo global utiliza todas las trayectorias disponibles.

Para una ejecución exploratoria puede usarse `RUN_MODE = "FAST"`. Para producir los resultados finales de \(L=200\), se usa `RUN_MODE = "GLOBAL"`.

**Mejoras incorporadas antes del entrenamiento global:** ruido y posición relativa compatibles con \(L=100\), pesos de pérdida revisados, umbrales ampliados, modo `GLOBAL_SHORT_TEST`, y matrices principales con errores normalizados (`nMAE_all`, `nRMSE_all`).


## 1.1 Connexion Google Drive pour Colab\n\nCette cellule monte Google Drive afin de lire les fichiers `.h5` depuis `MyDrive/TFM_L200`.\n

In [ ]:
# Google Drive is optional: enabled automatically in Colab, disabled locally.
import os
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

USE_GOOGLE_DRIVE = os.environ.get("TFM_USE_GOOGLE_DRIVE", "1" if IN_COLAB else "0") == "1"
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive mounting is only available in Colab")
    drive.mount("/content/drive")


In [ ]:
import json
import os
import random
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, losses, metrics, optimizers

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ============================================================
# MODE TEST RAPIDE
# ============================================================
RUN_MODE = os.environ.get("TFM_RUN_MODE", "FAST").upper()
FAST_RUN = RUN_MODE.upper() == "FAST"
GLOBAL_RUN = RUN_MODE.upper() == "GLOBAL"

GLOBAL_SHORT_TEST = os.environ.get("TFM_GLOBAL_SHORT_TEST", "0") == "1"
GLOBAL_SHORT_EPOCHS = int(os.environ.get("TFM_GLOBAL_SHORT_EPOCHS", "1"))

FAST_TRAIN_SIZE = 10_000
FAST_VAL_SIZE = 2_000
FAST_TEST_SIZE = 4_000

# ============================================================
# CONFIGURATION L = 200
# ============================================================
LENGTH = 200
DX_LENGTH = LENGTH - 1
MIN_SEGMENT_LENGTH = 40  # 20% de L=200, comparable con L=100

VALID_DX_MIN = MIN_SEGMENT_LENGTH - 1
VALID_DX_MAX = LENGTH - MIN_SEGMENT_LENGTH - 1
CP_RELATIVE_RANGE = (MIN_SEGMENT_LENGTH / LENGTH, (LENGTH - MIN_SEGMENT_LENGTH) / LENGTH)

MODEL_NAMES = ["ATTM", "CTRW", "FBM", "LW", "SBM"]
MODEL_MAP = {i: name for i, name in enumerate(MODEL_NAMES)}

TRANSITIONS = [(m1, m2) for m1 in MODEL_NAMES for m2 in MODEL_NAMES if m1 != m2]
TRANSITION_ORDER = [f"{m1} → {m2}" for m1, m2 in TRANSITIONS]

# ============================================================
# CHEMINS
# ============================================================
USE_GOOGLE_DRIVE = globals().get("USE_GOOGLE_DRIVE", False)
LOCAL_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DEFAULT_PROJECT_ROOT = Path("/content/drive/MyDrive/TFM_L200") if USE_GOOGLE_DRIVE else LOCAL_PROJECT_ROOT
PROJECT_ROOT = Path(os.environ.get("TFM_PROJECT_ROOT", str(DEFAULT_PROJECT_ROOT))).expanduser()
DATA_DIR = PROJECT_ROOT / "data_synthetic_with_without_changepoint_dx"

OUTPUT_DIR = PROJECT_ROOT / "model_outputs" / (
    "convtransformer_L200_fast_detection_localization_dx"
    if FAST_RUN
    else "convtransformer_L200_global_detection_localization_dx"
)

FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# PARAMÈTRES D'ENTRAÎNEMENT POUR TEST
# ============================================================
BATCH_SIZE = 128
MAX_EPOCHS = (
    GLOBAL_SHORT_EPOCHS
    if GLOBAL_RUN and GLOBAL_SHORT_TEST
    else (3 if FAST_RUN else 100)
)

LEARNING_RATE = 1e-3
LOSS_WEIGHTS = [2.0, 1.0]

THRESHOLDS = np.round(np.arange(0.05, 0.501, 0.025), 3)

MONITOR_METRIC = "val_selection_score"

DATA_FILES = {
    "train": DATA_DIR / f"train_L{LENGTH}_dim1_with_without_dx.h5",
    "val": DATA_DIR / f"val_L{LENGTH}_dim1_with_without_dx.h5",
    "test": DATA_DIR / f"test_L{LENGTH}_dim1_with_without_dx.h5",
}

if FAST_RUN == GLOBAL_RUN:
    raise ValueError("Choose exactly one mode: FAST_RUN=True or GLOBAL_RUN=True.")

pd.DataFrame({
    "item": [
        "RUN_MODE",
        "FAST_RUN",
        "GLOBAL_RUN",
        "GLOBAL_SHORT_TEST",
        "LENGTH",
        "DX_LENGTH",
        "VALID_CP_RANGE",
        "VALID_DX_RANGE",
        "OUTPUT_DIR",
        "BATCH_SIZE",
        "MAX_EPOCHS",
        "LOSS_WEIGHTS",
        "THRESHOLDS_MIN_MAX",
    ],
    "value": [
        RUN_MODE,
        FAST_RUN,
        GLOBAL_RUN,
        GLOBAL_SHORT_TEST,
        LENGTH,
        DX_LENGTH,
        f"{MIN_SEGMENT_LENGTH}–{LENGTH - MIN_SEGMENT_LENGTH}",
        f"{VALID_DX_MIN}–{VALID_DX_MAX}",
        str(OUTPUT_DIR),
        BATCH_SIZE,
        MAX_EPOCHS,
        str(LOSS_WEIGHTS),
        f"{THRESHOLDS.min()}–{THRESHOLDS.max()} ({len(THRESHOLDS)} valores)",
    ],
})

## 1.2 Vérification Colab : chemins et GPU\n\nAvant de charger les données, on vérifie que les fichiers `.h5` existent dans Google Drive et que le GPU est actif.\n

In [ ]:
# Vérification Colab : fichiers et GPU
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR =", DATA_DIR)

for name, path in DATA_FILES.items():
    print(name, path, "exists =", path.exists())

import tensorflow as tf
print("GPU devices:", tf.config.list_physical_devices("GPU"))


## 2. Carga de datos experimentales

Se leen las particiones HDF5 generadas en el cuaderno de construcción del conjunto sintético. En esta fase, la entrada utilizada por el modelo es `dx(t)`, por lo que la posición absoluta `x(t)` no se incorpora directamente al entrenamiento.

La comprobación inicial se limita a verificar la disponibilidad y coherencia básica de las variables necesarias, ya que la validación estructural del conjunto de datos se documenta en el cuaderno correspondiente.


In [ ]:

def decode_model(value):
    if isinstance(value, bytes):
        text = value.decode("utf-8")
        return text
    if isinstance(value, str):
        return value
    return MODEL_MAP[int(value)]


def normalize_dx(dx):
    mean = dx.mean(axis=1, keepdims=True)
    std = dx.std(axis=1, keepdims=True)
    return (dx - mean) / np.maximum(std, 1e-6)


def load_split(split_name):
    file_path = DATA_FILES[split_name]
    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {file_path}")

    with h5py.File(file_path, "r") as file:
        dx = file["dx"][:].astype("float32")
        has_cp = file["has_changepoint"][:].astype("float32")
        cp = file["cp"][:].astype("int16")
        cp_dx = file["cp_dx"][:].astype("int16") if "cp_dx" in file else (cp - 1).astype("int16")
        model1 = file["model1"][:]
        model2 = file["model2"][:]
        noise_sigma = file["noise_sigma"][:] if "noise_sigma" in file else np.full(len(has_cp), np.nan, dtype="float32")
        cp_relative_position = file["cp_relative_position"][:] if "cp_relative_position" in file else np.where(cp > 0, cp / LENGTH, np.nan).astype("float32")

    if dx.ndim == 2:
        dx = dx[:, :, None]

    dx = normalize_dx(dx).astype("float32")
    cp_class = np.where(has_cp == 1, cp_dx, 0).astype("int32")

    metadata = pd.DataFrame({
        "split": split_name,
        "has_changepoint": has_cp.astype(int),
        "cp": cp,
        "cp_dx": cp_dx,
        "cp_class": cp_class,
        "model1": [decode_model(value) for value in model1],
        "model2": [decode_model(value) for value in model2],
        "noise_sigma": noise_sigma.astype(float),
        "cp_relative_position": cp_relative_position.astype(float),
    })

    metadata["transition"] = np.where(
        metadata["has_changepoint"] == 1,
        metadata["model1"] + " → " + metadata["model2"],
        metadata["model1"] + " (no changepoint)",
    )

    return dx, has_cp.reshape(-1, 1).astype("float32"), cp_class.astype("int32"), metadata


x_train_full, y_has_train_full, y_cp_train_full, train_metadata_full = load_split("train")
x_val_full, y_has_val_full, y_cp_val_full, val_metadata_full = load_split("val")
x_test_full, y_has_test_full, y_cp_test_full, test_metadata_full = load_split("test")

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_examples": [len(x_train_full), len(x_val_full), len(x_test_full)],
    "dx_shape": [x_train_full.shape[1:], x_val_full.shape[1:], x_test_full.shape[1:]],
})


# Comprobación específica para L=200.
assert x_train_full.shape[1] == DX_LENGTH, f"Train dx length esperado {DX_LENGTH}, obtenido {x_train_full.shape[1]}"
assert x_val_full.shape[1] == DX_LENGTH, f"Val dx length esperado {DX_LENGTH}, obtenido {x_val_full.shape[1]}"
assert x_test_full.shape[1] == DX_LENGTH, f"Test dx length esperado {DX_LENGTH}, obtenido {x_test_full.shape[1]}"
print(f"Datos L={LENGTH} cargados correctamente: dx_length={DX_LENGTH}")


## 2.1 Preuve de la data corrigée\n\nCette cellule vérifie que la data L=200 contient le bruit, le changepoint relatif 20–80 % et les formes attendues.\n

In [ ]:
# Preuve que la data L=200 corrigée est bien utilisée
def summarize_loaded_split(name, x, y_has, metadata):
    mask_cp = metadata["has_changepoint"].to_numpy() == 1
    print("\n==============================")
    print(name.upper())
    print("n_examples:", len(metadata))
    print("dx shape:", x.shape)
    print("has_changepoint counts:")
    print(metadata["has_changepoint"].value_counts().sort_index())
    print("noise_sigma values:", sorted(np.round(metadata["noise_sigma"].dropna().unique(), 3)))
    print("cp min/max with changepoint:", int(metadata.loc[mask_cp, "cp"].min()), int(metadata.loc[mask_cp, "cp"].max()))
    print("cp_dx min/max with changepoint:", int(metadata.loc[mask_cp, "cp_dx"].min()), int(metadata.loc[mask_cp, "cp_dx"].max()))
    print("relative cp min/max:", float(metadata.loc[mask_cp, "cp_relative_position"].min()), float(metadata.loc[mask_cp, "cp_relative_position"].max()))
    print("number of transitions with changepoint:", metadata.loc[mask_cp, "transition"].nunique())

    assert x.shape[1:] == (DX_LENGTH, 1)
    assert set(np.round(metadata["noise_sigma"].dropna().unique(), 1)) == {0.1, 0.5, 1.0}
    assert metadata.loc[mask_cp, "cp"].min() >= 40
    assert metadata.loc[mask_cp, "cp"].max() <= 160

summarize_loaded_split("train", x_train_full, y_has_train_full, train_metadata_full)
summarize_loaded_split("validation", x_val_full, y_has_val_full, val_metadata_full)
summarize_loaded_split("test", x_test_full, y_has_test_full, test_metadata_full)

print("\nVerification completed successfully: nouvelle data L=200 avec bruit et cp relatif 20–80%.")


## 3. Selección del régimen de ejecución

El modo rápido selecciona un subconjunto equilibrado de trayectorias con y sin punto de cambio para validar la ejecución completa del cuaderno. El modo global emplea cada partición en su totalidad y se reserva para la evaluación experimental final.

Esta separación permite distinguir entre una comprobación funcional del procedimiento y el entrenamiento completo destinado a producir resultados comparables entre arquitecturas.

In [ ]:

def balanced_subset(x, y_has, y_cp_class, metadata, n_total, seed):
    rng = np.random.default_rng(seed)
    n_with = n_total // 2
    n_without = n_total - n_with

    with_meta = metadata[metadata["has_changepoint"] == 1]
    without_meta = metadata[metadata["has_changepoint"] == 0]

    selected = []

    per_transition = n_with // len(TRANSITION_ORDER)
    extra_transition = n_with % len(TRANSITION_ORDER)

    for i, transition in enumerate(TRANSITION_ORDER):
        group = with_meta[with_meta["transition"] == transition].index.to_numpy()
        k = per_transition + (1 if i < extra_transition else 0)
        selected.extend(rng.choice(group, size=min(k, len(group)), replace=False))

    per_model = n_without // len(MODEL_NAMES)
    extra_model = n_without % len(MODEL_NAMES)

    for i, model_name in enumerate(MODEL_NAMES):
        group = without_meta[without_meta["model1"] == model_name].index.to_numpy()
        k = per_model + (1 if i < extra_model else 0)
        selected.extend(rng.choice(group, size=min(k, len(group)), replace=False))

    selected = np.array(selected, dtype=int)
    rng.shuffle(selected)

    return (
        x[selected],
        y_has[selected],
        y_cp_class[selected],
        metadata.loc[selected].reset_index(drop=True),
    )


if FAST_RUN:
    x_train, y_has_train, y_cp_train, train_metadata = balanced_subset(
        x_train_full, y_has_train_full, y_cp_train_full, train_metadata_full, FAST_TRAIN_SIZE, RANDOM_SEED
    )
    x_val, y_has_val, y_cp_val, val_metadata = balanced_subset(
        x_val_full, y_has_val_full, y_cp_val_full, val_metadata_full, FAST_VAL_SIZE, RANDOM_SEED + 1
    )
    x_test, y_has_test, y_cp_test, test_metadata = balanced_subset(
        x_test_full, y_has_test_full, y_cp_test_full, test_metadata_full, FAST_TEST_SIZE, RANDOM_SEED + 2
    )
else:
    x_train, y_has_train, y_cp_train, train_metadata = x_train_full, y_has_train_full, y_cp_train_full, train_metadata_full
    x_val, y_has_val, y_cp_val, val_metadata = x_val_full, y_has_val_full, y_cp_val_full, val_metadata_full
    x_test, y_has_test, y_cp_test, test_metadata = x_test_full, y_has_test_full, y_cp_test_full, test_metadata_full

summary_rows = []
for name, x, y_has, metadata in [
    ("train", x_train, y_has_train, train_metadata),
    ("validation", x_val, y_has_val, val_metadata),
    ("test", x_test, y_has_test, test_metadata),
]:
    counts = metadata["has_changepoint"].value_counts().sort_index()
    summary_rows.append({
        "split": name,
        "n_examples": len(x),
        "dx_shape": x.shape[1:],
        "no_changepoint": int(counts.get(0, 0)),
        "changepoint": int(counts.get(1, 0)),
        "n_available_transitions": int(metadata[metadata["has_changepoint"] == 1]["transition"].nunique()),
    })

pd.DataFrame(summary_rows)


## 3.1 Aleatorización reproducible del conjunto de entrenamiento

El conjunto de entrenamiento puede estar organizado por bloques según la clase o el modelo de difusión. Si los lotes sucesivos son demasiado homogéneos, el proceso de optimización puede verse sesgado aunque el equilibrio global sea correcto.

Para evitarlo, se aplica una permutación aleatoria reproducible únicamente sobre el conjunto de entrenamiento antes de construir el objeto `tf.data.Dataset`. Las particiones de validación y test se mantienen sin mezclar para conservar una evaluación estable y comparable.

In [ ]:

rng = np.random.default_rng(RANDOM_SEED)
train_permutation = rng.permutation(len(x_train))

x_train = x_train[train_permutation]
y_has_train = y_has_train[train_permutation]
y_cp_train = y_cp_train[train_permutation]
train_metadata = train_metadata.iloc[train_permutation].reset_index(drop=True)

print("Global train label distribution after shuffle:")
print(pd.Series(y_has_train.reshape(-1)).value_counts())

print("First 20k train labels after shuffle:")
print(pd.Series(y_has_train[:20000].reshape(-1)).value_counts())

print("Middle 20k train labels after shuffle:")
mid = len(y_has_train) // 2
print(pd.Series(y_has_train[mid:mid+20000].reshape(-1)).value_counts())

print("Last 20k train labels after shuffle:")
print(pd.Series(y_has_train[-20000:].reshape(-1)).value_counts())


## 4. Definición de etiquetas y ponderación de pérdidas

La salida `has_cp` se define para todas las trayectorias y permite entrenar la detección binaria. La salida `cp_dist` solo tiene significado físico cuando existe un punto de cambio real, por lo que la pérdida de localización se pondera con valor cero en las trayectorias sin punto de cambio.

Esta formulación evita imponer una posición artificial a trayectorias homogéneas y separa explícitamente el aprendizaje de la presencia del cambio de la estimación de su ubicación temporal.

In [ ]:

def make_sample_weights(y_has):
    y_flat = y_has.reshape(-1).astype("float32")
    return (
        np.ones_like(y_flat, dtype="float32"),
        y_flat.astype("float32"),
    )


def make_dataset(x, y_has, y_cp_class, training=False):
    y = (
        y_has.astype("float32"),
        y_cp_class.astype("int32"),
    )
    sample_weight = make_sample_weights(y_has)
    dataset = tf.data.Dataset.from_tensor_slices((x, y, sample_weight))
    if training:
        dataset = dataset.shuffle(min(10_000, len(x)), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(x_train, y_has_train, y_cp_train, training=True)
val_ds = make_dataset(x_val, y_has_val, y_cp_val, training=False)
test_ds = make_dataset(x_test, y_has_test, y_cp_test, training=False)

print("Datasets ready.")
print("Output order: [has_cp, cp_dist]")


## 5. Estrategia de ponderación y selección fina de umbrales

La detección binaria requiere equilibrar dos riesgos: clasificar como cambiantes trayectorias homogéneas y no detectar transiciones reales. Para abordar este compromiso, el entrenamiento incorpora una ponderación específica de la clase sin punto de cambio y una búsqueda de umbral más detallada en la zona de decisión relevante.

Los valores definidos en el código, como `negative_weight`, `LOSS_WEIGHTS` y la malla de umbrales, forman parte de la configuración experimental. Su función es ajustar el equilibrio entre la tasa de falsos positivos y la sensibilidad de detección, manteniendo la salida `cp_dist` como representación temporal del punto de cambio.

## 6. Arquitectura ConvTransformer

El modelo incorpora un bloque convolucional 1D antes de los módulos de atención. Las convoluciones extraen patrones locales en `dx(t)`, mientras que la atención multi-cabeza integra relaciones temporales de mayor alcance.

La salida `has_cp` estima la probabilidad de presencia de punto de cambio mediante una activación sigmoide. La salida `cp_dist` produce una distribución `softmax` sobre las posiciones temporales válidas y se utiliza para la localización temporal.

In [ ]:

@tf.keras.utils.register_keras_serializable(package="Changepoint")
class ValidPositionMask(layers.Layer):
    def __init__(self, sequence_length, valid_min, valid_max, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = int(sequence_length)
        self.valid_min = int(valid_min)
        self.valid_max = int(valid_max)

    def call(self, logits):
        positions = tf.range(self.sequence_length)
        valid = tf.logical_and(positions >= self.valid_min, positions <= self.valid_max)
        valid = tf.cast(valid, tf.float32)
        mask = (1.0 - valid) * (-1e9)
        return logits + mask[tf.newaxis, :]

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "valid_min": self.valid_min,
            "valid_max": self.valid_max,
        })
        return config


@tf.keras.utils.register_keras_serializable(package="Changepoint")
class LastStep(layers.Layer):
    def call(self, inputs):
        return inputs[:, -1, :]


@tf.keras.utils.register_keras_serializable(package="Changepoint")
class SqueezeLastAxis(layers.Layer):
    def call(self, inputs):
        return tf.squeeze(inputs, axis=-1)


@tf.keras.utils.register_keras_serializable(package="Changepoint")
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, d_model, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = int(sequence_length)
        self.d_model = int(d_model)
        self.position_embedding = layers.Embedding(
            input_dim=self.sequence_length,
            output_dim=self.d_model,
            name="position_embedding_table",
        )

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        embedded_positions = self.position_embedding(positions)
        return inputs + embedded_positions[tf.newaxis, :, :]

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "d_model": self.d_model,
        })
        return config


@tf.keras.utils.register_keras_serializable(package="Changepoint")
def sparse_ce_localization(y_true, y_pred):
    y_true = tf.reshape(tf.cast(y_true, tf.int32), [-1])
    n_positions = tf.shape(y_pred)[1]
    y_true = tf.clip_by_value(y_true, 0, n_positions - 1)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    batch_size = tf.shape(y_pred)[0]
    gather_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), y_true], axis=1)
    true_prob = tf.gather_nd(y_pred, gather_indices)
    return -tf.math.log(true_prob)


@tf.keras.utils.register_keras_serializable(package="Changepoint")
def weighted_binary_crossentropy_no_cp(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

    positive_weight = tf.constant(1.0, dtype=tf.float32)
    negative_weight = tf.constant(1.0, dtype=tf.float32)

    positive_loss = -positive_weight * y_true * tf.math.log(y_pred)
    negative_loss = -negative_weight * (1.0 - y_true) * tf.math.log(1.0 - y_pred)

    return tf.reduce_mean(positive_loss + negative_loss)


def conv_stem_block(x, d_model, dropout_rate):
    """Local feature extractor before the attention blocks."""
    x = layers.Conv1D(
        filters=d_model // 2,
        kernel_size=5,
        padding="same",
        name="conv_stem_1",
    )(x)
    x = layers.LayerNormalization(epsilon=1e-6, name="conv_stem_norm_1")(x)
    x = layers.Activation("gelu", name="conv_stem_gelu_1")(x)
    x = layers.Dropout(dropout_rate, name="conv_stem_dropout_1")(x)

    x = layers.Conv1D(
        filters=d_model,
        kernel_size=3,
        padding="same",
        name="conv_stem_2",
    )(x)
    x = layers.LayerNormalization(epsilon=1e-6, name="conv_stem_norm_2")(x)
    x = layers.Activation("gelu", name="conv_stem_gelu_2")(x)
    x = layers.Dropout(dropout_rate, name="conv_stem_dropout_2")(x)
    return x


def conv_residual_block(x, d_model, kernel_size, dropout_rate, block_id, dilation_rate=1):
    """Residual Conv1D block that preserves the temporal length."""
    residual = x

    y = layers.Conv1D(
        filters=d_model,
        kernel_size=kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        name=f"conv_residual_{block_id}_conv_1",
    )(x)
    y = layers.LayerNormalization(epsilon=1e-6, name=f"conv_residual_{block_id}_norm_1")(y)
    y = layers.Activation("gelu", name=f"conv_residual_{block_id}_gelu_1")(y)
    y = layers.Dropout(dropout_rate, name=f"conv_residual_{block_id}_dropout_1")(y)

    y = layers.Conv1D(
        filters=d_model,
        kernel_size=kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        name=f"conv_residual_{block_id}_conv_2",
    )(y)
    y = layers.Dropout(dropout_rate, name=f"conv_residual_{block_id}_dropout_2")(y)

    x = layers.Add(name=f"conv_residual_{block_id}_add")([residual, y])
    x = layers.LayerNormalization(epsilon=1e-6, name=f"conv_residual_{block_id}_norm_2")(x)
    return x


def transformer_encoder_block(x, d_model, num_heads, ff_dim, dropout_rate, block_id):
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout_rate,
        name=f"transformer_{block_id}_mha",
    )(x, x)
    attention_output = layers.Dropout(dropout_rate, name=f"transformer_{block_id}_attn_dropout")(attention_output)
    x = layers.Add(name=f"transformer_{block_id}_attn_add")([x, attention_output])
    x = layers.LayerNormalization(epsilon=1e-6, name=f"transformer_{block_id}_attn_norm")(x)

    ffn = layers.Dense(ff_dim, activation="gelu", name=f"transformer_{block_id}_ffn_dense_1")(x)
    ffn = layers.Dropout(dropout_rate, name=f"transformer_{block_id}_ffn_dropout_1")(ffn)
    ffn = layers.Dense(d_model, name=f"transformer_{block_id}_ffn_dense_2")(ffn)
    ffn = layers.Dropout(dropout_rate, name=f"transformer_{block_id}_ffn_dropout_2")(ffn)
    x = layers.Add(name=f"transformer_{block_id}_ffn_add")([x, ffn])
    x = layers.LayerNormalization(epsilon=1e-6, name=f"transformer_{block_id}_ffn_norm")(x)
    return x


def build_convtransformer_detection_localization_model(
    input_shape=(DX_LENGTH, 1),
    d_model=96,
    num_heads=4,
    ff_dim=192,
    num_transformer_blocks=3,
    num_conv_blocks=2,
    conv_kernel_size=5,
    dropout_rate=0.15,
):
    inputs = layers.Input(shape=input_shape, name="dx_input")

    # ConvTransformer change: local Conv1D feature extraction before attention.
    x = conv_stem_block(inputs, d_model=d_model, dropout_rate=dropout_rate)

    for conv_id in range(1, num_conv_blocks + 1):
        dilation_rate = 1 if conv_id == 1 else 2
        x = conv_residual_block(
            x=x,
            d_model=d_model,
            kernel_size=conv_kernel_size,
            dropout_rate=dropout_rate,
            block_id=conv_id,
            dilation_rate=dilation_rate,
        )

    x = PositionalEmbedding(input_shape[0], d_model, name="positional_embedding")(x)
    x = layers.Dropout(dropout_rate, name="embedding_dropout")(x)

    for block_id in range(1, num_transformer_blocks + 1):
        x = transformer_encoder_block(
            x=x,
            d_model=d_model,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout_rate=dropout_rate,
            block_id=block_id,
        )

    avg_pool = layers.GlobalAveragePooling1D(name="average_pooling")(x)
    max_pool = layers.GlobalMaxPooling1D(name="max_pooling")(x)
    last_state = LastStep(name="last_state")(x)

    shared = layers.Concatenate(name="shared_features")([avg_pool, max_pool, last_state])
    shared = layers.Dense(128, activation="gelu", name="shared_dense_1")(shared)
    shared = layers.Dropout(0.25, name="shared_dropout_1")(shared)
    shared = layers.Dense(64, activation="gelu", name="shared_dense_2")(shared)

    det = layers.Dense(64, activation="gelu", name="detection_dense_1")(shared)
    det = layers.Dropout(0.20, name="detection_dropout")(det)
    has_cp = layers.Dense(1, activation="sigmoid", name="has_cp")(det)

    loc = layers.Dense(96, activation="gelu", name="localization_dense_1")(x)
    loc = layers.Dropout(0.15, name="localization_dropout")(loc)
    loc = layers.Dense(48, activation="gelu", name="localization_dense_2")(loc)
    logits = layers.Dense(1, name="localization_logits_dense")(loc)
    logits = SqueezeLastAxis(name="localization_logits")(logits)
    logits = ValidPositionMask(DX_LENGTH, VALID_DX_MIN, VALID_DX_MAX, name="valid_position_mask")(logits)
    cp_dist = layers.Activation("softmax", name="cp_dist")(logits)

    return models.Model(inputs=inputs, outputs=[has_cp, cp_dist], name="convtransformer_detection_localization_dx")


model = build_convtransformer_detection_localization_model(input_shape=(DX_LENGTH, 1))

model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=[
        weighted_binary_crossentropy_no_cp,
        sparse_ce_localization,
    ],
    loss_weights=LOSS_WEIGHTS,
    metrics=[
        [
            metrics.BinaryAccuracy(name="accuracy"),
            metrics.Precision(name="precision"),
            metrics.Recall(name="recall"),
        ],
        [
            metrics.SparseCategoricalAccuracy(name="sparse_accuracy"),
        ],
    ],
)

model.summary()


## 7. Métricas de validación y selección del umbral

Durante el entrenamiento se calcula un criterio de validación que combina calidad de detección, control de falsos positivos y precisión de localización temporal. El umbral final se selecciona a partir del conjunto de validación, no como un valor fijo independiente de los datos.

Esta selección permite ajustar la sensibilidad del modelo ante trayectorias con punto de cambio y su robustez frente a trayectorias homogéneas.


In [ ]:

def unpack_predictions(predictions):
    if isinstance(predictions, (list, tuple)):
        return predictions[0], predictions[1]
    return predictions["has_cp"], predictions["cp_dist"]


def detection_metrics(y_true, probabilities, threshold):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    probabilities = np.asarray(probabilities).reshape(-1)
    y_pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0)),
        "false_positive_rate": float(fp / (fp + tn + 1e-8)),
        "false_negative_rate": float(fn / (fn + tp + 1e-8)),
        "jaccard_coefficient": float(tp / (tp + fp + fn + 1e-8)),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    }


def soft_positions(cp_dist):
    positions = np.arange(cp_dist.shape[1], dtype=np.float32)
    return np.sum(cp_dist * positions.reshape(1, -1), axis=1) + 1.0


def localization_metrics(metadata, probabilities, cp_dist, threshold):
    y_true_has = metadata["has_changepoint"].to_numpy(dtype=int)
    y_pred_has = (np.asarray(probabilities).reshape(-1) >= threshold).astype(int)
    pred_cp = soft_positions(cp_dist)

    mask_all = y_true_has == 1
    mask_tp = (y_true_has == 1) & (y_pred_has == 1)

    def compute(mask, prefix):
        if int(mask.sum()) == 0:
            return {f"{prefix}_n": 0, f"{prefix}_mae": np.nan, f"{prefix}_rmse": np.nan}
        true_cp = metadata.loc[mask, "cp"].to_numpy(dtype=np.float32)
        return {
            f"{prefix}_n": int(mask.sum()),
            f"{prefix}_mae": float(mean_absolute_error(true_cp, pred_cp[mask])),
            f"{prefix}_rmse": float(np.sqrt(mean_squared_error(true_cp, pred_cp[mask]))),
        }

    values = {}
    values.update(compute(mask_all, "all_changepoint"))
    values.update(compute(mask_tp, "true_positive"))
    return values


def threshold_table(y_true, probabilities, cp_dist, metadata, thresholds=THRESHOLDS):
    rows = []
    for threshold in thresholds:
        row = detection_metrics(y_true, probabilities, threshold)
        row.update(localization_metrics(metadata, probabilities, cp_dist, threshold))
        rows.append(row)
    return pd.DataFrame(rows)


def choose_threshold(table):
    candidates = table[
        (table["false_positive_rate"] <= 0.35) &
        (table["recall"] >= 0.65)
    ].copy()

    if candidates.empty:
        candidates = table[
            (table["false_positive_rate"] <= 0.45) &
            (table["recall"] >= 0.60)
        ].copy()

    if candidates.empty:
        candidates = table.copy()

    candidates = candidates.sort_values(
        ["f1_score", "false_positive_rate", "precision"],
        ascending=[False, True, False],
    )

    return candidates.iloc[0].to_dict()


class ValidationMetricsCallback(callbacks.Callback):
    def __init__(self, x_val, y_has_val, val_metadata, batch_size=1024):
        super().__init__()
        self.x_val = x_val
        self.y_has_val = y_has_val.reshape(-1).astype(int)
        self.val_metadata = val_metadata.reset_index(drop=True)
        self.batch_size = batch_size
        self.records = []
        self.threshold_records = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        has_prob, cp_dist = unpack_predictions(self.model.predict(self.x_val, batch_size=self.batch_size, verbose=0))
        has_prob = has_prob.reshape(-1)
        cp_dist = np.asarray(cp_dist)

        table = threshold_table(self.y_has_val, has_prob, cp_dist, self.val_metadata, THRESHOLDS)
        best = choose_threshold(table)
        score = (
            best["all_changepoint_mae"]
            + 20.0 * (1.0 - best["f1_score"])
            + 18.0 * best["false_positive_rate"]
            + 10.0 * max(0.0, 0.70 - best["recall"])
        )

        record = {
            "epoch": epoch + 1,
            "val_best_threshold": float(best["threshold"]),
            "val_detection_accuracy": float(best["accuracy"]),
            "val_detection_precision": float(best["precision"]),
            "val_detection_recall": float(best["recall"]),
            "val_detection_f1": float(best["f1_score"]),
            "val_false_positive_rate": float(best["false_positive_rate"]),
            "val_false_negative_rate": float(best["false_negative_rate"]),
            "val_jaccard_coefficient": float(best["jaccard_coefficient"]),
            "val_localization_mae_all_cp": float(best["all_changepoint_mae"]),
            "val_localization_rmse_all_cp": float(best["all_changepoint_rmse"]),
            "val_localization_mae_true_positive": float(best["true_positive_mae"]),
            "val_localization_rmse_true_positive": float(best["true_positive_rmse"]),
            "val_selection_score": float(score),
        }

        self.records.append(record)
        table["epoch"] = epoch + 1
        self.threshold_records.append(table)
        logs.update({k: v for k, v in record.items() if k != "epoch"})

        print(
            f"\nval_threshold={record['val_best_threshold']:.3f} | "
            f"F1={record['val_detection_f1']:.4f} | "
            f"recall={record['val_detection_recall']:.4f} | "
            f"FPR={record['val_false_positive_rate']:.4f} | "
            f"RMSE_TP={record['val_localization_rmse_true_positive']:.2f}"
        )


val_metrics_callback = ValidationMetricsCallback(x_val, y_has_val, val_metadata)


### Nota antes de ejecutar `GLOBAL`

Antes de lanzar el entrenamiento completo, ejecuta primero:

```python
RUN_MODE = "FAST"
```

Después, puedes hacer un control corto del modo global:

```python
RUN_MODE = "GLOBAL"
GLOBAL_SHORT_TEST = True
GLOBAL_SHORT_EPOCHS = 1
```

Solo cuando esta ejecución de una época termine sin errores se recomienda usar:

```python
GLOBAL_SHORT_TEST = False
```


## 8. Estrategia de entrenamiento y almacenamiento local

El entrenamiento utiliza mecanismos de seguimiento para registrar métricas, guardar el mejor modelo, ajustar la tasa de aprendizaje y detener la optimización cuando la validación deja de mejorar. Los artefactos generados durante este proceso se guardan localmente y no deben incorporarse al repositorio.

La opción `GLOBAL_SHORT_TEST` permite validar la ejecución global de forma abreviada antes de lanzar la configuración completa.

In [ ]:

checkpoint_path = OUTPUT_DIR / "best_convtransformer_detection_localization_dx.keras"
training_log_path = OUTPUT_DIR / "training_log_convtransformer_detection_localization_dx.csv"

training_callbacks = [
    val_metrics_callback,
    callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor=MONITOR_METRIC,
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor=MONITOR_METRIC,
        patience=8 if FAST_RUN else 15,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor=MONITOR_METRIC,
        factor=0.5,
        patience=4 if FAST_RUN else 6,
        mode="min",
        min_lr=1e-6,
        verbose=1,
    ),
    callbacks.CSVLogger(training_log_path),
]

print("Active mode:", "FAST_RUN" if FAST_RUN else "GLOBAL_RUN")
print("Global short test:", GLOBAL_SHORT_TEST)
print("Output directory:", OUTPUT_DIR)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=training_callbacks,
    verbose=1,
)


## 9. Evaluación posterior al entrenamiento

Después del entrenamiento se recarga el mejor modelo según el desempeño de validación. La selección del umbral se realiza con la partición de validación y la evaluación final se calcula sobre el conjunto de test.

Este procedimiento separa la optimización, la selección del criterio de decisión y la estimación final del rendimiento.

In [ ]:

history_frame = pd.DataFrame(history.history)
validation_metrics_frame = pd.DataFrame(val_metrics_callback.records)
threshold_history_frame = pd.concat(val_metrics_callback.threshold_records, ignore_index=True)

history_frame.to_csv(OUTPUT_DIR / "history_convtransformer_detection_localization_dx.csv", index=False)
validation_metrics_frame.to_csv(OUTPUT_DIR / "validation_metrics_convtransformer_detection_localization_dx.csv", index=False)
threshold_history_frame.to_csv(OUTPUT_DIR / "threshold_history_convtransformer_detection_localization_dx.csv", index=False)

best_model = tf.keras.models.load_model(
    checkpoint_path,
    custom_objects={
        "ValidPositionMask": ValidPositionMask,
        "LastStep": LastStep,
        "SqueezeLastAxis": SqueezeLastAxis,
        "PositionalEmbedding": PositionalEmbedding,
        "sparse_ce_localization": sparse_ce_localization,
        "weighted_binary_crossentropy_no_cp": weighted_binary_crossentropy_no_cp,
    },
    compile=False,
    safe_mode=False,
)

def predict_outputs(model_to_use, x):
    has_prob, cp_dist = unpack_predictions(model_to_use.predict(x, batch_size=BATCH_SIZE, verbose=1))
    return np.asarray(has_prob).reshape(-1), np.asarray(cp_dist)


val_has_prob, val_cp_dist = predict_outputs(best_model, x_val)
test_has_prob, test_cp_dist = predict_outputs(best_model, x_test)

validation_threshold_table = threshold_table(y_has_val, val_has_prob, val_cp_dist, val_metadata, THRESHOLDS)
test_threshold_table = threshold_table(y_has_test, test_has_prob, test_cp_dist, test_metadata, THRESHOLDS)

selected_threshold = float(choose_threshold(validation_threshold_table)["threshold"])
validation_threshold_table["selected"] = validation_threshold_table["threshold"] == selected_threshold
test_threshold_table["selected"] = test_threshold_table["threshold"] == selected_threshold

validation_threshold_table.to_csv(OUTPUT_DIR / "threshold_comparison_validation.csv", index=False)
test_threshold_table.to_csv(OUTPUT_DIR / "threshold_comparison_test.csv", index=False)

print("Threshold Comparison for Changepoint Detection — Validation")
display(validation_threshold_table)

print("Threshold Comparison for Changepoint Detection — Test")
display(test_threshold_table)

print("Selected threshold:", selected_threshold)


## 10. Métricas globales y matriz de confusión

Las métricas principales se calculan en el conjunto de test. La evaluación combina indicadores de detección binaria, métricas de localización temporal y una matriz de confusión para resumir la separación entre trayectorias con y sin punto de cambio.

El análisis conjunto permite valorar tanto la capacidad del modelo para reconocer cambios reales como su tendencia a generar falsas alarmas.

In [ ]:

def save_current_figure(name):
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=170, bbox_inches="tight")
    print("Saved figure:", path)


selected_test = test_threshold_table[test_threshold_table["threshold"] == selected_threshold].iloc[0].to_dict()
test_pred_has = (test_has_prob >= selected_threshold).astype(int)
test_true_has = y_has_test.reshape(-1).astype(int)
test_cm = confusion_matrix(test_true_has, test_pred_has, labels=[0, 1])

global_summary = pd.DataFrame([
    {"metric": "selected_threshold", "value": selected_threshold},
    {"metric": "accuracy", "value": selected_test["accuracy"]},
    {"metric": "precision", "value": selected_test["precision"]},
    {"metric": "recall", "value": selected_test["recall"]},
    {"metric": "f1_score", "value": selected_test["f1_score"]},
    {"metric": "false_positive_rate", "value": selected_test["false_positive_rate"]},
    {"metric": "false_negative_rate", "value": selected_test["false_negative_rate"]},
    {"metric": "jaccard_coefficient", "value": selected_test["jaccard_coefficient"]},
    {"metric": "mae_all_changepoint", "value": selected_test["all_changepoint_mae"]},
    {"metric": "rmse_all_changepoint", "value": selected_test["all_changepoint_rmse"]},
    {"metric": "mae_true_positive", "value": selected_test["true_positive_mae"]},
    {"metric": "rmse_true_positive", "value": selected_test["true_positive_rmse"]},
])

global_summary.to_csv(OUTPUT_DIR / "global_test_summary.csv", index=False)
global_summary


In [ ]:

plt.figure(figsize=(5, 4))
plt.imshow(test_cm, aspect="auto")
plt.xticks([0, 1], ["No changepoint", "Changepoint"])
plt.yticks([0, 1], ["No changepoint", "Changepoint"])
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Confusion Matrix for Changepoint Detection")
for i in range(test_cm.shape[0]):
    for j in range(test_cm.shape[1]):
        plt.text(j, i, str(test_cm[i, j]), ha="center", va="center")
plt.colorbar()
plt.tight_layout()
save_current_figure("confusion_matrix_for_changepoint_detection.png")
plt.show()


## 11. Visualización del entrenamiento y del criterio de decisión

Las figuras de esta sección presentan la evolución de las pérdidas, las métricas de detección y la comparación entre los umbrales evaluados.

Estas visualizaciones permiten interpretar la estabilidad del entrenamiento y el efecto del umbral elegido sobre el equilibrio entre sensibilidad y control de falsos positivos.


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(history_frame["loss"], label="Training loss")
plt.plot(history_frame["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
save_current_figure("training_and_validation_loss.png")
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(validation_metrics_frame["val_detection_precision"], label="Validation precision")
plt.plot(validation_metrics_frame["val_detection_recall"], label="Validation recall")
plt.plot(validation_metrics_frame["val_detection_f1"], label="Validation F1-score")
plt.plot(validation_metrics_frame["val_jaccard_coefficient"], label="Validation Jaccard coefficient")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Detection Metrics Across Epochs")
plt.legend()
plt.grid(alpha=0.3)
save_current_figure("detection_metrics_across_epochs.png")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(validation_threshold_table["threshold"], validation_threshold_table["precision"], marker="o", label="Precision")
plt.plot(validation_threshold_table["threshold"], validation_threshold_table["recall"], marker="o", label="Recall")
plt.plot(validation_threshold_table["threshold"], validation_threshold_table["f1_score"], marker="o", label="F1-score")
plt.plot(validation_threshold_table["threshold"], validation_threshold_table["false_positive_rate"], marker="o", label="False positive rate")
plt.axvline(selected_threshold, linestyle="--", label=f"Selected threshold = {selected_threshold:.2f}")
plt.xlabel("Decision threshold for has_cp")
plt.ylabel("Score")
plt.title("Threshold Comparison for Changepoint Detection")
plt.legend()
plt.grid(alpha=0.3)
save_current_figure("threshold_comparison_for_changepoint_detection.png")
plt.show()


## 12. Análisis por transición ordenada para \(L=200\)

El análisis por transición descompone el rendimiento según el par ordenado de modelos de difusión que define cada trayectoria con punto de cambio. Esta lectura permite identificar qué cambios de dinámica son más accesibles para la arquitectura ConvTransformer.

La información por transición complementa las métricas globales y ayuda a interpretar cómo el modelo responde a distintas combinaciones de difusión anómala dentro de la misma formulación de detección binaria y localización temporal.

In [ ]:

test_pred_cp = soft_positions(test_cp_dist)

results_frame = test_metadata.copy()
results_frame["has_cp_prob"] = test_has_prob
results_frame["has_cp_pred"] = test_pred_has
results_frame["pred_cp"] = test_pred_cp
results_frame["abs_error"] = np.where(
    results_frame["has_changepoint"] == 1,
    np.abs(results_frame["pred_cp"] - results_frame["cp"]),
    np.nan,
)
results_frame["squared_error"] = np.where(
    results_frame["has_changepoint"] == 1,
    (results_frame["pred_cp"] - results_frame["cp"]) ** 2,
    np.nan,
)

transition_rows = []
with_cp = results_frame[results_frame["has_changepoint"] == 1].copy()

for transition in TRANSITION_ORDER:
    group = with_cp[with_cp["transition"] == transition].copy()
    if group.empty:
        continue
    tp_group = group[group["has_cp_pred"] == 1].copy()
    transition_rows.append({
        "transition": transition,
        "model1": group["model1"].iloc[0],
        "model2": group["model2"].iloc[0],
        "n_examples": int(len(group)),
        "n_true_positives": int((group["has_cp_pred"] == 1).sum()),
        "n_false_negatives": int((group["has_cp_pred"] == 0).sum()),
        "detection_recall": float((group["has_cp_pred"] == 1).mean()),
        "false_negative_rate": float((group["has_cp_pred"] == 0).mean()),
        "mae_all": float(group["abs_error"].mean()),
        "rmse_all": float(np.sqrt(group["squared_error"].mean())),
        "mae_true_positive": float(tp_group["abs_error"].mean()) if len(tp_group) else np.nan,
        "rmse_true_positive": float(np.sqrt(tp_group["squared_error"].mean())) if len(tp_group) else np.nan,
    })

transition_metrics = pd.DataFrame(transition_rows)

# Errores normalizados por la longitud de la trayectoria para comparar L=100 y L=200.
transition_metrics["nmae_all"] = transition_metrics["mae_all"] / LENGTH
transition_metrics["nrmse_all"] = transition_metrics["rmse_all"] / LENGTH
transition_metrics["nmae_true_positive"] = transition_metrics["mae_true_positive"] / LENGTH
transition_metrics["nrmse_true_positive"] = transition_metrics["rmse_true_positive"] / LENGTH

transition_metrics.to_csv(OUTPUT_DIR / "transition_level_metrics.csv", index=False)
transition_metrics.sort_values("rmse_true_positive").head(10)


In [ ]:
def transition_matrix(column):
    matrix = pd.DataFrame(np.nan, index=MODEL_NAMES, columns=MODEL_NAMES, dtype=float)
    for _, row in transition_metrics.iterrows():
        matrix.loc[row["model1"], row["model2"]] = row[column]
    return matrix


def plot_transition_matrix(matrix, title, colorbar_label, filename, value_format="{:.2f}", vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix.values, aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(MODEL_NAMES)))
    ax.set_yticks(np.arange(len(MODEL_NAMES)))
    ax.set_xticklabels(MODEL_NAMES)
    ax.set_yticklabels(MODEL_NAMES)
    ax.set_xlabel("Modelo de la segunda parte de la trayectoria")
    ax.set_ylabel("Modelo de la primera parte de la trayectoria")
    ax.set_title(title)

    for i in range(len(MODEL_NAMES)):
        for j in range(len(MODEL_NAMES)):
            value = matrix.iloc[i, j]
            text = "—" if np.isnan(value) else value_format.format(value)
            ax.text(j, i, text, ha="center", va="center")

    plt.colorbar(image, ax=ax, label=colorbar_label)
    plt.tight_layout()
    save_current_figure(filename)
    plt.show()


recall_matrix = transition_matrix("detection_recall")
fnr_matrix = transition_matrix("false_negative_rate")
nmae_all_matrix = transition_matrix("nmae_all")
nrmse_all_matrix = transition_matrix("nrmse_all")

plot_transition_matrix(
    recall_matrix,
    "ConvTransformer L=200 — Recall por transición",
    "Recall de detección",
    "L200_convtransformer_recall_by_transition_matrix.png",
    value_format="{:.2f}",
    vmin=0,
    vmax=1,
)

plot_transition_matrix(
    nrmse_all_matrix,
    "ConvTransformer L=200 — nRMSE_all por transición",
    "RMSE_all / L",
    "L200_convtransformer_nrmse_all_by_transition_matrix.png",
    value_format="{:.3f}",
    vmin=0,
    vmax=0.30,
)


## 13. Identificación de transiciones de distinta dificultad

Se calcula una medida de dificultad que combina la tasa de falsos negativos y el error de localización. Las transiciones con menor sensibilidad o mayor error temporal se consideran más difíciles.

Esta ordenación facilita la selección de casos representativos y la comparación posterior con otras arquitecturas.


In [ ]:

def minmax(series):
    series = series.astype(float)
    if series.max() == series.min():
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / (series.max() - series.min())


difficulty = transition_metrics.copy()
difficulty["rmse_for_score"] = difficulty["nrmse_all"]
difficulty["difficulty_score"] = minmax(difficulty["false_negative_rate"]) + minmax(difficulty["rmse_for_score"])

easy = difficulty.sort_values("difficulty_score").head(5).copy()
difficult = difficulty.sort_values("difficulty_score", ascending=False).head(5).copy()

easy["category"] = "Easy"
difficult["category"] = "Difficult"

easy_difficult = pd.concat([easy, difficult], ignore_index=True)
easy_difficult["interpretation"] = np.where(
    easy_difficult["category"] == "Easy",
    "High detection recall and low localization error.",
    "Low detection recall and/or high localization error.",
)

easy_difficult = easy_difficult[
    [
        "category",
        "transition",
        "detection_recall",
        "false_negative_rate",
        "mae_all",
        "rmse_all",
        "nmae_all",
        "nrmse_all",
        "mae_true_positive",
        "rmse_true_positive",
        "interpretation",
    ]
]

easy_difficult.to_csv(OUTPUT_DIR / "easy_difficult_transitions.csv", index=False)
easy_difficult


In [ ]:

plot_data = difficulty.copy()
plt.figure(figsize=(8, 6))
plt.scatter(plot_data["detection_recall"], plot_data["rmse_for_score"])

for _, row in pd.concat([easy, difficult]).iterrows():
    plt.text(row["detection_recall"], row["rmse_for_score"], row["transition"], fontsize=8)

plt.xlabel("Recall de detección")
plt.ylabel("Localization RMSE in temporal points")
plt.title("Transition-Level Detection and Localization Difficulty")
plt.grid(alpha=0.3)
save_current_figure("transition_level_detection_localization_difficulty.png")
plt.show()


## 14. Ejemplos representativos de trayectorias

Se visualizan trayectorias asociadas a transiciones de distinta dificultad. La posición real del punto de cambio y la estimación derivada de `cp_dist` permiten comparar de forma cualitativa la referencia temporal con la predicción del modelo.

Estos ejemplos conectan el comportamiento numérico del modelo con patrones concretos observados en `dx(t)`.

In [ ]:

selected_transitions = easy["transition"].tolist()[:3] + difficult["transition"].tolist()[:3]
selected_indices = []

for transition in selected_transitions:
    group = results_frame[(results_frame["has_changepoint"] == 1) & (results_frame["transition"] == transition)].copy()
    tp_group = group[group["has_cp_pred"] == 1].copy()
    if len(tp_group):
        group = tp_group
    if len(group):
        median_error = group["abs_error"].median()
        selected_indices.append((group["abs_error"] - median_error).abs().idxmin())

n_cols = 2
n_rows = int(np.ceil(len(selected_indices) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.6 * n_rows), squeeze=False)
axes = axes.ravel()

for ax, idx in zip(axes, selected_indices):
    row = results_frame.loc[idx]
    ax.plot(x_test[idx, :, 0], linewidth=1.0, label="dx(t)")
    ax.axvline(row["cp_dx"], linestyle="--", linewidth=1.2, label="True changepoint")
    ax.axvline(row["pred_cp"] - 1.0, linestyle="-", linewidth=1.2, label="Predicted changepoint")
    ax.set_title(
        f"{row['transition']}\nabsolute error = {row['abs_error']:.1f} points | P(changepoint) = {row['has_cp_prob']:.2f}",
        fontsize=9,
    )
    ax.set_xlabel("Position in dx(t)")
    ax.set_ylabel("Normalized increment")
    ax.grid(alpha=0.25)

for j in range(len(selected_indices), len(axes)):
    axes[j].axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")
fig.suptitle("Representative Trajectories with True and Predicted Changepoints", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
save_current_figure("representative_trajectories_true_predicted_changepoints.png")
plt.show()


## 15. Evaluación complementaria del modelo

Las siguientes celdas amplían el análisis del mejor modelo guardado mediante visualizaciones adicionales. Se estudian falsos negativos, falsos positivos, errores de localización por transición y dependencia del rendimiento respecto a la posición temporal real del punto de cambio.

Estas salidas no modifican el entrenamiento ni la arquitectura; su finalidad es interpretar con mayor detalle el comportamiento del ConvTransformer frente a trayectorias con y sin punto de cambio.

In [ ]:

def build_transition_matrix(metric_name):
    matrix = pd.DataFrame(np.nan, index=MODEL_NAMES, columns=MODEL_NAMES, dtype=float)
    for _, row in transition_metrics.iterrows():
        matrix.loc[row["model1"], row["model2"]] = row[metric_name]
    return matrix


def save_matrix_csv(matrix, filename):
    path = OUTPUT_DIR / filename
    matrix.to_csv(path)
    return path


def plot_matrix(matrix, title, colorbar_label, filename, value_format="{:.2f}", vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix.values, aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(MODEL_NAMES)))
    ax.set_yticks(np.arange(len(MODEL_NAMES)))
    ax.set_xticklabels(MODEL_NAMES)
    ax.set_yticklabels(MODEL_NAMES)
    ax.set_xlabel("Modelo de la segunda parte de la trayectoria")
    ax.set_ylabel("Modelo de la primera parte de la trayectoria")
    ax.set_title(title)

    for i in range(len(MODEL_NAMES)):
        for j in range(len(MODEL_NAMES)):
            value = matrix.iloc[i, j]
            text = "—" if np.isnan(value) else value_format.format(value)
            ax.text(j, i, text, ha="center", va="center")

    plt.colorbar(image, ax=ax, label=colorbar_label)
    plt.tight_layout()
    save_current_figure(filename)
    plt.show()


### 15.1 Tasa de falsos negativos por transición — \(L=200\)

Esta matriz muestra, para cada transición real, la proporción de trayectorias con punto de cambio clasificadas como trayectorias sin punto de cambio. Los valores altos indican transiciones cuya detección resulta especialmente difícil para el modelo.

El análisis permite localizar patrones de error que no siempre son visibles en las métricas globales.

In [ ]:

false_negative_matrix = build_transition_matrix("false_negative_rate")
save_matrix_csv(false_negative_matrix, "false_negative_rate_by_transition.csv")

plot_matrix(
    false_negative_matrix,
    "ConvTransformer L=200 — FNR por transición",
    "Tasa de falsos negativos",
    "L200_convtransformer_false_negative_rate_by_transition_matrix.png",
    value_format="{:.2f}",
    vmin=0,
    vmax=1,
)

false_negative_matrix


### 15.2 Tasa de falsos positivos en trayectorias sin punto de cambio

Esta tabla y su figura identifican qué modelos homogéneos generan con mayor frecuencia predicciones incorrectas de punto de cambio.

El resultado permite evaluar si determinadas dinámicas de difusión anómala producen fluctuaciones que el ConvTransformer interpreta como indicios de transición, aun cuando la trayectoria no contiene un punto de cambio real.

In [ ]:

no_changepoint_results = results_frame[results_frame["has_changepoint"] == 0].copy()

false_positive_by_model = (
    no_changepoint_results
    .groupby("model1")
    .agg(
        total_no_changepoint=("has_cp_pred", "size"),
        false_positives=("has_cp_pred", "sum"),
        false_positive_rate=("has_cp_pred", "mean"),
    )
    .reindex(MODEL_NAMES)
    .reset_index()
    .rename(columns={"model1": "diffusion_model"})
)

false_positive_by_model.to_csv(OUTPUT_DIR / "false_positive_rate_by_no_changepoint_model.csv", index=False)

plt.figure(figsize=(7, 4))
plt.bar(false_positive_by_model["diffusion_model"], false_positive_by_model["false_positive_rate"])
plt.ylim(0, 1)
plt.xlabel("No-changepoint diffusion model")
plt.ylabel("False positive rate")
plt.title("False Positive Rate by No-Changepoint Diffusion Model")
plt.grid(axis="y", alpha=0.3)
save_current_figure("false_positive_rate_by_no_changepoint_model.png")
plt.show()

false_positive_by_model


### 15.3 Errores MAE y RMSE de localización por transición — \(L=200\)

Se comparan el error absoluto medio (`MAE`) y la raíz del error cuadrático medio (`RMSE`) para cada transición con punto de cambio. El `MAE` describe el error típico, mientras que el `RMSE` permite detectar la presencia de errores temporales de gran magnitud.

Esta comparación aporta una lectura más completa de la localización temporal que una sola métrica agregada.

In [ ]:

mae_transition_matrix = build_transition_matrix("mae_all")
rmse_transition_matrix = build_transition_matrix("rmse_all")
nmae_transition_matrix = build_transition_matrix("nmae_all")
nrmse_transition_matrix = build_transition_matrix("nrmse_all")

save_matrix_csv(mae_transition_matrix, "localization_mae_by_transition.csv")
save_matrix_csv(rmse_transition_matrix, "localization_rmse_all_by_transition.csv")
save_matrix_csv(nmae_transition_matrix, "localization_nmae_all_by_transition.csv")
save_matrix_csv(nrmse_transition_matrix, "localization_nrmse_all_by_transition.csv")

plot_matrix(
    mae_transition_matrix,
    "ConvTransformer L=200 — MAE de localización por transición",
    "MAE de localización en puntos temporales",
    "L200_convtransformer_localization_mae_by_transition_matrix.png",
    value_format="{:.1f}",
)

plot_matrix(
    rmse_transition_matrix,
    "ConvTransformer L=200 — RMSE de localización por transición",
    "RMSE de localización en puntos temporales",
    "L200_convtransformer_localization_rmse_all_by_transition_matrix.png",
    value_format="{:.1f}",
)

mae_transition_matrix


plot_matrix(
    nmae_transition_matrix,
    "ConvTransformer L=200 — nMAE_all por transición",
    "MAE_all / L",
    "L200_convtransformer_localization_nmae_all_by_transition_matrix.png",
    value_format="{:.3f}",
    vmin=0,
    vmax=0.25,
)

plot_matrix(
    nrmse_transition_matrix,
    "ConvTransformer L=200 — nRMSE_all por transición",
    "RMSE_all / L",
    "L200_convtransformer_localization_nrmse_all_by_transition_matrix.png",
    value_format="{:.3f}",
    vmin=0,
    vmax=0.30,
)


### Figura matricial resumida para la indicación del profesor

La siguiente figura agrupa tres lecturas complementarias para \(L=200\): tasa de falsos negativos, MAE de localización y RMSE de localización. En todos los paneles, las filas representan el modelo de la primera parte de la trayectoria y las columnas el modelo de la segunda parte.


In [ ]:

def annotate_matrix(ax, matrix, value_format):
    for i in range(len(MODEL_NAMES)):
        for j in range(len(MODEL_NAMES)):
            value = matrix.iloc[i, j]
            text = "—" if np.isnan(value) else value_format.format(value)
            ax.text(j, i, text, ha="center", va="center", fontsize=8)

professor_matrices = [
    (false_negative_matrix, "FNR", "{:.2f}", 0, 1),
    (nmae_transition_matrix, "nMAE_all", "{:.3f}", 0, 0.25),
    (nrmse_transition_matrix, "nRMSE_all", "{:.3f}", 0, 0.30),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)

for ax, (matrix, title, value_format, vmin, vmax) in zip(axes, professor_matrices):
    image = ax.imshow(matrix.values, aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(MODEL_NAMES)))
    ax.set_yticks(np.arange(len(MODEL_NAMES)))
    ax.set_xticklabels(MODEL_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(MODEL_NAMES)
    ax.set_xlabel("Modelo de la segunda parte")
    ax.set_ylabel("Modelo de la primera parte")
    ax.set_title(title)
    annotate_matrix(ax, matrix, value_format)
    plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    "ConvTransformer L=200: matrices por transición ordenada "
    "(filas = primer modelo, columnas = segundo modelo)",
    fontsize=14,
)
save_current_figure("L200_convtransformer_matrices_transicion_profesor_normalizadas.png")
plt.show()


### 15.4 Ordenación de transiciones según dificultad

La clasificación combina errores de detección y de localización para ordenar las transiciones de menor a mayor dificultad. Las transiciones con alta sensibilidad y bajo error temporal ocupan las posiciones más favorables.

Esta ordenación resulta útil para interpretar el comportamiento del modelo y seleccionar ejemplos cualitativos representativos.

In [ ]:

transition_ranking = transition_metrics.copy()
transition_ranking["localization_rmse"] = transition_ranking["rmse_all"]
transition_ranking["localization_mae"] = transition_ranking["mae_all"]

def normalized_series(values):
    values = values.astype(float)
    if values.max() == values.min():
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - values.min()) / (values.max() - values.min())

transition_ranking["normalized_false_negative_rate"] = normalized_series(transition_ranking["false_negative_rate"])
transition_ranking["normalized_localization_rmse"] = normalized_series(transition_ranking["localization_rmse"])
transition_ranking["difficulty_score"] = (
    transition_ranking["normalized_false_negative_rate"] +
    transition_ranking["normalized_localization_rmse"]
)

ranked_transitions = transition_ranking.sort_values("difficulty_score").reset_index(drop=True)
ranked_transitions["rank"] = np.arange(1, len(ranked_transitions) + 1)
ranked_transitions["interpretation"] = np.where(
    ranked_transitions["rank"] <= 5,
    "Easy transition: high detection recall and low localization error.",
    np.where(
        ranked_transitions["rank"] > len(ranked_transitions) - 5,
        "Difficult transition: low detection recall and/or high localization error.",
        "Intermediate transition."
    )
)

transition_difficulty_ranking = ranked_transitions[
    [
        "rank",
        "transition",
        "detection_recall",
        "false_negative_rate",
        "localization_mae",
        "localization_rmse",
        "difficulty_score",
        "interpretation",
    ]
]

transition_difficulty_ranking.to_csv(OUTPUT_DIR / "transition_difficulty_ranking.csv", index=False)

top_easiest = transition_difficulty_ranking.head(5)
top_difficult = transition_difficulty_ranking.tail(5).sort_values("difficulty_score", ascending=False)

plt.figure(figsize=(9, 5))
plot_table = pd.concat([top_easiest, top_difficult])
plt.barh(plot_table["transition"], plot_table["difficulty_score"])
plt.xlabel("Difficulty score")
plt.ylabel("Transition")
plt.title("Ranking of Transition Difficulty")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
save_current_figure("transition_difficulty_ranking.png")
plt.show()

print("Top 5 easiest transitions")
display(top_easiest)

print("Top 5 most difficult transitions")
display(top_difficult)



### 15.5 Error de localización según intervalo temporal del punto de cambio

Las trayectorias con punto de cambio se agrupan según la zona temporal en la que aparece la transición real. Esta evaluación permite analizar si el modelo localiza mejor los cambios situados en la zona central o si pierde precisión cerca de los extremos.

La dependencia temporal es relevante porque el contexto disponible antes y después del cambio puede afectar a la estimación.

In [ ]:

changepoint_results = results_frame[results_frame["has_changepoint"] == 1].copy()

min_cp = int(np.floor(changepoint_results["cp"].min() / 10) * 10)
max_cp = int(np.ceil(changepoint_results["cp"].max() / 10) * 10)
bins = np.arange(min_cp, max_cp + 10, 10)

if len(bins) < 3:
    bins = np.linspace(changepoint_results["cp"].min(), changepoint_results["cp"].max() + 1, 4)

labels = [f"{int(bins[i])}–{int(bins[i+1])}" for i in range(len(bins) - 1)]
changepoint_results["cp_interval"] = pd.cut(
    changepoint_results["cp"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False,
)

interval_rows = []

for interval, group in changepoint_results.groupby("cp_interval", observed=False):
    if len(group) == 0:
        continue

    tp_group = group[group["has_cp_pred"] == 1]
    y_true_interval = np.ones(len(group), dtype=int)
    y_pred_interval = group["has_cp_pred"].to_numpy(dtype=int)

    interval_rows.append({
        "true_changepoint_interval": str(interval),
        "number_of_examples": int(len(group)),
        "detection_recall": float(group["has_cp_pred"].mean()),
        "detection_f1_score": float(f1_score(y_true_interval, y_pred_interval, zero_division=0)),
        "localization_mae": float(group["abs_error"].mean()),
        "localization_rmse": float(np.sqrt(group["squared_error"].mean())),
        "localization_mae_true_positive": float(tp_group["abs_error"].mean()) if len(tp_group) else np.nan,
        "localization_rmse_true_positive": float(np.sqrt(tp_group["squared_error"].mean())) if len(tp_group) else np.nan,
    })

interval_performance = pd.DataFrame(interval_rows)
interval_performance.to_csv(OUTPUT_DIR / "changepoint_interval_performance.csv", index=False)

x_positions = np.arange(len(interval_performance))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(x_positions, interval_performance["localization_rmse"], marker="o", label="All changepoints")
axes[0].plot(x_positions, interval_performance["localization_rmse_true_positive"], marker="s", label="True positives")
axes[0].set_title("Localization RMSE by True Changepoint Interval")
axes[0].set_xlabel("True changepoint interval")
axes[0].set_ylabel("RMSE in time points")
axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(interval_performance["true_changepoint_interval"], rotation=45)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(x_positions, interval_performance["localization_mae"], marker="o", label="All changepoints")
axes[1].plot(x_positions, interval_performance["localization_mae_true_positive"], marker="s", label="True positives")
axes[1].set_title("Localization MAE by True Changepoint Interval")
axes[1].set_xlabel("True changepoint interval")
axes[1].set_ylabel("MAE in time points")
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(interval_performance["true_changepoint_interval"], rotation=45)
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(x_positions, interval_performance["detection_f1_score"], marker="o", label="Detection F1-score")
axes[2].plot(x_positions, interval_performance["detection_recall"], marker="s", label="Recall de detección")
axes[2].set_title("Detection Performance by True Changepoint Interval")
axes[2].set_xlabel("True changepoint interval")
axes[2].set_ylabel("Score")
axes[2].set_ylim(0, 1.05)
axes[2].set_xticks(x_positions)
axes[2].set_xticklabels(interval_performance["true_changepoint_interval"], rotation=45)
axes[2].legend()
axes[2].grid(alpha=0.3)

fig.suptitle("Performance by True Changepoint Interval", fontsize=15)
plt.tight_layout()
save_current_figure("performance_by_true_changepoint_interval.png")
plt.show()

interval_performance


### 15.6 Relación entre MAE y RMSE por transición

La figura compara `MAE` y `RMSE` por transición. Cuando el `RMSE` es considerablemente mayor que el `MAE`, la transición puede contener errores aislados de gran magnitud.

Esta lectura ayuda a distinguir entre errores distribuidos de forma regular y fallos puntuales con fuerte impacto en la métrica cuadrática.


In [ ]:

relationship_data = transition_metrics.copy()
relationship_data["relationship_mae"] = relationship_data["mae_true_positive"].fillna(relationship_data["mae_all"])
relationship_data["relationship_rmse"] = relationship_data["rmse_true_positive"].fillna(relationship_data["rmse_all"])

plt.figure(figsize=(8, 6))
plt.scatter(relationship_data["relationship_mae"], relationship_data["relationship_rmse"])

for _, row in top_difficult.iterrows():
    transition_name = row["transition"]
    point = relationship_data[relationship_data["transition"] == transition_name].iloc[0]
    plt.text(point["relationship_mae"], point["relationship_rmse"], transition_name, fontsize=8)

plt.xlabel("ConvTransformer L=200 — MAE de localización por transición")
plt.ylabel("ConvTransformer L=200 — RMSE de localización por transición")
plt.title("Relationship Between MAE and RMSE by Transition")
plt.grid(alpha=0.3)
save_current_figure("mae_rmse_relationship_by_transition.png")
plt.show()


### 15.7 Rendimiento en función de la posición temporal del punto de cambio

Esta visualización resume cómo cambia el rendimiento según la posición real del punto de cambio dentro de la trayectoria. El objetivo es detectar posibles sesgos temporales en la capacidad de detección o localización.

Un comportamiento estable a lo largo de los intervalos temporales indicaría que el modelo aprovecha de manera equilibrada la información disponible en la secuencia.


In [ ]:

fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(
    interval_performance["true_changepoint_interval"],
    interval_performance["localization_rmse_true_positive"],
    marker="o",
    label="Localization RMSE on true positives",
)
ax1.plot(
    interval_performance["true_changepoint_interval"],
    interval_performance["localization_mae_true_positive"],
    marker="s",
    label="Localization MAE on true positives",
)
ax1.set_xlabel("True changepoint interval")
ax1.set_ylabel("Localization error in time points")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(
    interval_performance["true_changepoint_interval"],
    interval_performance["detection_recall"],
    marker="^",
    linestyle="--",
    label="Recall de detección",
)
ax2.set_ylabel("Recall de detección")
ax2.set_ylim(0, 1.05)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper center")

plt.title("Changepoint Position-Dependent Performance")
plt.tight_layout()
save_current_figure("changepoint_position_dependent_performance.png")
plt.show()


### 15.8 Interpretación intermedia de la evaluación complementaria

La evaluación complementaria separa distintos tipos de error para facilitar la interpretación del ConvTransformer. La tasa de falsos negativos identifica transiciones reales que no se detectan de forma consistente, mientras que la tasa de falsos positivos muestra qué trayectorias homogéneas producen alertas incorrectas.

El análisis de `MAE`, `RMSE` y posición temporal del punto de cambio permite estudiar la localización desde una perspectiva más detallada. Esta sección funciona como lectura intermedia de las visualizaciones anteriores y reserva la síntesis cuantitativa del modelo para el resumen final.

## Resumen final

Al ejecutar este cuaderno se generará el resumen final del experimento ConvTransformer para \(L=200\). Los valores exactos se guardarán automáticamente en:

- `global_test_summary_convtransformer_detection_localization_dx.csv`
- `transition_level_metrics.csv`
- `transition_difficulty_ranking.csv`
- `final_summary_convtransformer_detection_localization_dx.json`

Las figuras matriciales solicitadas por el profesor se guardarán en el subdirectorio `figures/`, especialmente:

- `L200_convtransformer_false_negative_rate_by_transition_matrix.png`
- `L200_convtransformer_localization_mae_by_transition_matrix.png`
- `L200_convtransformer_localization_rmse_by_transition_matrix.png`
- `L200_convtransformer_matrices_transicion_profesor.png`


In [ ]:

final_summary = {
    "mode": "FAST_RUN" if FAST_RUN else "GLOBAL_RUN",
    "train_samples": int(x_train.shape[0]),
    "validation_samples": int(x_val.shape[0]),
    "test_samples": int(x_test.shape[0]),
    "selected_threshold": selected_threshold,
    "loss_weights": LOSS_WEIGHTS,
    "global_test_summary": global_summary.to_dict(orient="records"),
    "output_dir": str(OUTPUT_DIR),
}

with open(OUTPUT_DIR / "final_summary_convtransformer_detection_localization_dx.json", "w", encoding="utf-8") as file:
    json.dump(final_summary, file, indent=2, ensure_ascii=False)

best_model.save(OUTPUT_DIR / "convtransformer_detection_localization_dx_best_model.keras")

print("Evaluation outputs saved in:", OUTPUT_DIR)
print("Figures saved in:", FIGURES_DIR)

print("Resumen global seleccionado para ConvTransformer L=200:")
display(global_summary)

print("Top 5 transiciones más fáciles:")
display(top_easiest)

print("Top 5 transiciones más difíciles:")
display(top_difficult)


## Texto breve para la memoria o para responder al profesor

Una vez ejecutado el cuaderno, puede describirse el análisis de la siguiente forma:

> Para \(L=200\), se fijó la arquitectura ConvTransformer y se evaluaron las transiciones ordenadas entre los modelos ATTM, CTRW, FBM, LW y SBM. En las matrices resultantes, las filas corresponden al modelo generador de la primera parte de la trayectoria y las columnas al modelo generador de la segunda parte. Esta representación permite identificar qué pares de modelos concentran los principales problemas de detección y localización, más allá de las métricas globales del modelo.
